# 🇧🇩 Multi-Tool AI Agent for Bangladesh — Colab Walkthrough

This notebook is a **self-contained** version of the full project: it writes
every project file to disk with `%%writefile`, then runs the pipeline
step-by-step so you can see the output of each stage as you go.

**What you'll do, in order:**
1. Install dependencies
2. Set your API keys (via Colab Secrets or a plain prompt)
3. Recreate the project's folder structure and files
4. Download + inspect the 3 Hugging Face datasets
5. Build the 3 SQLite databases
6. Build the LangGraph routing agent
7. Ask it the 5 example questions and see which tool it picks

> Once you've pushed this project to GitHub, you can replace Step 3 below
> with a single `!git clone <your-repo-url>` cell instead — this notebook
> just doesn't assume a repo exists yet.


## Step 1 — Install dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-openai langgraph \
    datasets huggingface_hub pandas SQLAlchemy \
    tavily-python duckduckgo-search python-dotenv


## Step 2 — Set your API keys

**Required:** `OPENAI_API_KEY`
**Optional:** `TAVILY_API_KEY` (if blank, the web search tool automatically
falls back to DuckDuckGo, which needs no key at all)

If you're on Colab, the safest way is the **Secrets** panel (key icon 🔑 in
the left sidebar) — add `OPENAI_API_KEY` and `TAVILY_API_KEY` there, then
run the cell below. If a secret isn't found, it falls back to a hidden
`getpass` prompt so this also works outside Colab.


In [ ]:
import os

def get_secret(name):
    # Try Colab's Secrets manager first.
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # Fall back to a hidden prompt (works in Colab, Jupyter, or plain Python).
    from getpass import getpass
    val = getpass(f"Enter {name} (leave blank to skip): ")
    return val or ""

os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = get_secret("TAVILY_API_KEY")

assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY is required for this notebook to run."
print("Keys set. Tavily search:", "ENABLED" if os.environ["TAVILY_API_KEY"] else "using DuckDuckGo fallback (no key)")


## Step 3 — Recreate the project structure

Each `%%writefile` cell below writes one real project file to disk, exactly
as it exists in the full repo. Running these cells builds the same
`tools/`, `agent/`, and `scripts/` folders you'd get from cloning the repo.


In [ ]:
import os
for d in ["data", "db", "tools", "scripts", "agent"]:
    os.makedirs(d, exist_ok=True)
open("tools/__init__.py", "w").close()
open("agent/__init__.py", "w").close()
print("Folders created.")


### `tools/sql_safety.py` — the read-only SQL safety layer

In [ ]:
%%writefile tools/sql_safety.py
"""
Shared safety layer used by all three DB tools.

WHY THIS EXISTS
----------------
Our tools let an LLM generate SQL from a natural-language question and then
execute it. That's powerful, but also risky if the generated SQL is a stray
DROP TABLE, DELETE, or UPDATE. This module enforces two hard rules before
ANY query touches the database:

  1. Only a single SELECT statement is allowed (no semicolons chaining a
     second statement, no INSERT/UPDATE/DELETE/DROP/ALTER/ATTACH/etc.).
  2. A LIMIT is added automatically if the query doesn't already have one,
     so a broad question can't dump an enormous table into the LLM's
     context window.

We also open every SQLite connection in read-only mode as a second layer
of defense, using SQLite's URI "mode=ro" so writes fail at the database
engine level even if a bad query somehow got past the regex check.
"""

import re
import sqlite3

FORBIDDEN_KEYWORDS = [
    "INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "REPLACE",
    "ATTACH", "DETACH", "PRAGMA", "VACUUM", "TRUNCATE", "GRANT", "REVOKE",
]

DEFAULT_ROW_LIMIT = 200


class UnsafeSQLError(Exception):
    """Raised when generated SQL fails the safety check."""


def validate_select_only(sql: str) -> str:
    """
    Validates that `sql` is a single, read-only SELECT statement.
    Returns the (possibly LIMIT-appended) safe SQL string, or raises
    UnsafeSQLError.
    """
    cleaned = sql.strip().rstrip(";")

    if ";" in cleaned:
        raise UnsafeSQLError("Multiple SQL statements are not allowed.")

    if not re.match(r"^\s*SELECT\b", cleaned, flags=re.IGNORECASE):
        raise UnsafeSQLError("Only SELECT statements are allowed.")

    upper = cleaned.upper()
    for word in FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{word}\b", upper):
            raise UnsafeSQLError(f"Keyword '{word}' is not allowed in this tool.")

    if not re.search(r"\bLIMIT\b", upper):
        cleaned = f"{cleaned} LIMIT {DEFAULT_ROW_LIMIT}"

    return cleaned


def run_read_only_query(db_path: str, sql: str):
    """
    Opens the SQLite file in read-only URI mode and executes a validated
    SELECT query. Returns (column_names, rows).
    """
    safe_sql = validate_select_only(sql)

    uri = f"file:{db_path}?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    try:
        cursor = conn.execute(safe_sql)
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()
        return columns, rows
    finally:
        conn.close()


def get_table_schema(db_path: str, table_name: str) -> str:
    """Returns the CREATE TABLE statement for `table_name` from `db_path`."""
    uri = f"file:{db_path}?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    try:
        row = conn.execute(
            "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
            (table_name,),
        ).fetchone()
        return row[0] if row else ""
    finally:
        conn.close()


### `tools/db_tools.py` — the 3 database tools

In [ ]:
%%writefile tools/db_tools.py
"""
STEP 4: DB-specific LangChain tools.

HOW EACH TOOL WORKS (natural language -> SQL -> safe execution -> natural
language answer):

  1. The main agent calls a tool like `hospitals_db_tool` with the user's
     plain-English question (e.g. "How many hospitals are in Dhaka?").
  2. The tool asks a small LLM to write ONE SQLite SELECT query against the
     table's *real* schema (pulled live from the .db file, not hardcoded).
  3. `tools/sql_safety.py` validates the generated SQL is read-only-safe,
     adds a row LIMIT if missing, and executes it against the SQLite file
     opened in read-only mode.
  4. The raw rows are handed back to the LLM one more time, asking it to
     phrase the result as a short natural-language answer.

Each of the 3 concrete tools below is a thin wrapper around this same
factory function, pointed at a different database + table + description.
The **description is what the agent's router reads** to decide which tool
fits a given question, so it explicitly lists the kinds of questions each
tool is good for and (importantly) what it is NOT for, to reduce mis-routing.
"""

import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from tools.sql_safety import run_read_only_query, get_table_schema, UnsafeSQLError

DB_DIR = os.path.join(os.path.dirname(__file__), "..", "db")

# A small, cheap, fast model is enough for NL-to-SQL translation and result
# summarizing — you don't need your most expensive model for this.
_sql_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def _generate_sql(question: str, schema_sql: str, table_name: str) -> str:
    prompt = f"""You are a SQLite expert. Given this table schema:

{schema_sql}

Write ONE single SQLite SELECT query that answers the user's question below.
Rules:
- Only query the table "{table_name}".
- Return ONLY the raw SQL, no markdown fences, no explanation.
- Never use INSERT, UPDATE, DELETE, DROP, ALTER, or any write statement.
- Use LIKE with wildcards for text/location matching (e.g. WHERE location LIKE '%Dhaka%')
  since location names in the data may have inconsistent spacing/case.

Question: {question}
SQL:"""
    response = _sql_llm.invoke(prompt)
    sql = response.content.strip()
    # Strip accidental markdown code fences if the model adds them anyway.
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


def _summarize_result(question: str, columns: list, rows: list) -> str:
    if not rows:
        return "I ran the query but found no matching records in the database."

    preview = rows[:50]  # keep the summarizing prompt small
    prompt = f"""The user asked: "{question}"

The SQL query returned these columns: {columns}
And these rows (showing up to 50): {preview}

Write a short, clear, natural-language answer to the user's question based
strictly on this data. If the data answers a count/list, state it plainly.
Do not invent any information that isn't in the rows above."""
    response = _sql_llm.invoke(prompt)
    return response.content.strip()


def _make_db_tool(db_filename: str, table_name: str):
    """
    Returns a function that runs the full NL -> SQL -> execute -> summarize
    pipeline for one specific database + table.
    """
    db_path = os.path.join(DB_DIR, db_filename)

    def _run(question: str) -> str:
        schema_sql = get_table_schema(db_path, table_name)
        if not schema_sql:
            return (
                f"Database error: table '{table_name}' not found in {db_filename}. "
                f"Did you run scripts/build_databases.py?"
            )

        sql = _generate_sql(question, schema_sql, table_name)

        try:
            columns, rows = run_read_only_query(db_path, sql)
        except UnsafeSQLError as e:
            return f"I couldn't safely run that query ({e}). Try rephrasing the question."
        except Exception as e:
            return f"The database query failed: {e}. Try rephrasing the question."

        return _summarize_result(question, columns, rows)

    return _run


# ---------------------------------------------------------------------------
# The three concrete tools the main agent will register.
# ---------------------------------------------------------------------------

_institutions_runner = _make_db_tool("institutions.db", "institutions")
_hospitals_runner = _make_db_tool("hospitals.db", "hospitals")
_restaurants_runner = _make_db_tool("restaurants.db", "restaurants")


@tool
def institutions_db_tool(question: str) -> str:
    """Use this tool for questions about EDUCATIONAL and GOVERNMENT
    INSTITUTIONS in Bangladesh: universities, colleges, schools, government
    offices/agencies — their names, locations, types, and similar structured
    details. Examples: "What government universities are in Sylhet?",
    "List private colleges in Dhaka." Do NOT use this for hospitals or
    restaurants, and do NOT use this for general knowledge questions like
    policy or history — use the web search tool for those instead."""
    return _institutions_runner(question)


@tool
def hospitals_db_tool(question: str) -> str:
    """Use this tool for questions about HOSPITALS in Bangladesh: hospital
    names, locations, number of beds, doctors, facilities, and similar
    structured details. Examples: "How many hospitals are in Dhaka?",
    "Which hospitals in Chittagong have more than 100 beds?". Do NOT use this
    for institutions or restaurants, and do NOT use this for general
    healthcare-policy questions (e.g. "What is DGHS?") — use the web search
    tool for those instead."""
    return _hospitals_runner(question)


@tool
def restaurants_db_tool(question: str) -> str:
    """Use this tool for questions about RESTAURANTS in Bangladesh:
    restaurant names, locations, cuisine type, and ratings. Examples:
    "List restaurants in Chittagong with a rating above 4.",
    "What Thai restaurants are in Dhaka?". Do NOT use this for hospitals or
    institutions, and do NOT use this for general food-culture questions
    (e.g. "What is a popular Bangladeshi dish?") — use the web search tool
    for those instead."""
    return _restaurants_runner(question)


### `tools/web_search_tool.py` — the general-knowledge tool

In [ ]:
%%writefile tools/web_search_tool.py
"""
STEP 5: General-knowledge Web Search tool.

We use Tavily as the primary search backend (it's built specifically for
LLM agents and returns clean, summarized results) if TAVILY_API_KEY is set
in .env. If it's NOT set, we automatically fall back to DuckDuckGo search,
which requires no API key at all — this keeps the project runnable for a
beginner with zero search API signups.
"""

import os
from langchain_core.tools import tool

_TAVILY_KEY = os.getenv("TAVILY_API_KEY", "").strip()

if _TAVILY_KEY:
    from langchain_community.tools.tavily_search import TavilySearchResults

    _search_backend = TavilySearchResults(max_results=5)

    def _run_search(query: str) -> str:
        results = _search_backend.invoke({"query": query})
        # results is a list of dicts with 'content' and 'url'
        formatted = "\n\n".join(
            f"- {r.get('content', '')} (source: {r.get('url', '')})" for r in results
        )
        return formatted or "No results found."

else:
    from langchain_community.tools import DuckDuckGoSearchRun

    _search_backend = DuckDuckGoSearchRun()

    def _run_search(query: str) -> str:
        return _search_backend.invoke(query)


@tool
def web_search_tool(query: str) -> str:
    """Use this tool for GENERAL KNOWLEDGE questions that are NOT answerable
    from the institutions, hospitals, or restaurants databases — for example
    definitions, government policy, history, culture, or "what is X"
    questions about Bangladesh. Examples: "What is the role of DGHS in
    Bangladesh?", "What is Bangladesh's national health policy?". Do NOT use
    this for questions asking to count, list, or filter specific
    institutions, hospitals, or restaurants — use the matching DB tool for
    those instead."""
    return _run_search(query)


### `scripts/download_and_inspect.py`

In [ ]:
%%writefile scripts/download_and_inspect.py
"""
STEP 2: Download the three Bangladesh datasets from Hugging Face and inspect
them.

WHY THIS SCRIPT EXISTS
-----------------------
We don't actually know the exact column names, data types, or how messy each
CSV is until we look at it. Rather than guessing column names (which would
break the moment the real data doesn't match), this script:

  1. Downloads each dataset using the `datasets` library.
  2. Converts it to a pandas DataFrame.
  3. Prints the columns, dtypes, row count, a data sample, and null counts.
  4. Saves a clean CSV copy into ./data/ so the next script (build_databases.py)
     has a stable local file to work from.

Run this FIRST, read its printed output carefully, and only then move on to
build_databases.py. If a dataset has a messy/unexpected column (e.g. mixed
Bangla/English text, or a numeric column stored as text with commas), the
"MESSY DATA CHECK" section below will flag it so you can decide how to clean
it in build_databases.py.
"""

import os
import pandas as pd
from datasets import load_dataset

# Where we'll save the raw CSVs after downloading them once.
DATA_DIR = os.path.join(os.path.dirname(__file__), "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

# The three Hugging Face dataset repo IDs from the project brief.
DATASETS = {
    "institutions": "Mahadih534/Institutional-Information-of-Bangladesh",
    "hospitals": "Mahadih534/all-bangladeshi-hospitals",
    "restaurants": "Mahadih534/Bangladeshi-Restaurant-Data",
}


def download_as_dataframe(hf_repo_id: str) -> pd.DataFrame:
    """
    Downloads a Hugging Face dataset and returns it as a pandas DataFrame.

    We ask for the 'train' split, which is the default/only split for most
    small CSV-based datasets on the Hub. If a dataset has a different split
    name, `load_dataset` will raise a clear error telling us the available
    splits, and we can adjust the split name here.
    """
    ds = load_dataset(hf_repo_id, split="train")
    return ds.to_pandas()


def inspect_dataframe(name: str, df: pd.DataFrame) -> None:
    """Prints a human-readable summary of a DataFrame's structure and quality."""
    print("\n" + "=" * 70)
    print(f"DATASET: {name}  |  rows={len(df)}  cols={len(df.columns)}")
    print("=" * 70)

    print("\n--- Columns & dtypes ---")
    print(df.dtypes)

    print("\n--- First 5 rows ---")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(df.head(5))

    print("\n--- MESSY DATA CHECK ---")
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    if len(cols_with_nulls) > 0:
        print("Columns with missing values:")
        print(cols_with_nulls)
    else:
        print("No missing values detected.")

    # Flag object/text columns that look like they should actually be numbers
    # (common issue: "1,234" or "1234 beds" stored as text instead of int).
    for col in df.select_dtypes(include="object").columns:
        sample_vals = df[col].dropna().astype(str).head(20)
        looks_numeric_ish = sample_vals.str.replace(",", "", regex=False).str.match(
            r"^\d+(\.\d+)?$"
        )
        if looks_numeric_ish.any() and not looks_numeric_ish.all():
            print(
                f"NOTE: column '{col}' is text but partially looks numeric — "
                f"inspect manually before casting to INTEGER/REAL."
            )

    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"NOTE: {duplicate_count} fully duplicated rows found.")


def main():
    for short_name, repo_id in DATASETS.items():
        print(f"\nDownloading '{repo_id}' ...")
        df = download_as_dataframe(repo_id)

        inspect_dataframe(short_name, df)

        out_path = os.path.join(DATA_DIR, f"{short_name}.csv")
        df.to_csv(out_path, index=False)
        print(f"\nSaved raw copy -> {out_path}")

    print("\nAll datasets downloaded and inspected. Review the printed output "
          "above, then run scripts/build_databases.py next.")


if __name__ == "__main__":
    main()


### `scripts/build_databases.py`

In [ ]:
%%writefile scripts/build_databases.py
"""
STEP 3: Convert the downloaded CSVs into SQLite databases.

WHAT THIS SCRIPT DOES
----------------------
For each of the three datasets, it:
  1. Reads the CSV saved by download_and_inspect.py.
  2. Cleans obviously messy values (trims whitespace, strips thousands-
     separators from number-looking text columns, normalizes empty strings
     to real NULLs).
  3. Lets pandas infer proper dtypes (int64 / float64 / object), which
     SQLAlchemy then maps to SQLite's INTEGER / REAL / TEXT correctly.
  4. Writes the DataFrame into its own SQLite database file using `to_sql`.
  5. Prints the final CREATE TABLE statement pulled straight from SQLite's
     own schema table, so you can see exactly what was created.

OUTPUT
------
  db/institutions.db   -> table "institutions"
  db/hospitals.db      -> table "hospitals"
  db/restaurants.db    -> table "restaurants"
"""

import os
import re
import sqlite3
import pandas as pd
from sqlalchemy import create_engine

DATA_DIR = os.path.join(os.path.dirname(__file__), "..", "data")
DB_DIR = os.path.join(os.path.dirname(__file__), "..", "db")
os.makedirs(DB_DIR, exist_ok=True)

# short_name -> (csv filename, sqlite db filename, table name)
TABLES = {
    "institutions": ("institutions.csv", "institutions.db", "institutions"),
    "hospitals": ("hospitals.csv", "hospitals.db", "hospitals"),
    "restaurants": ("restaurants.csv", "restaurants.db", "restaurants"),
}


def clean_column_name(col: str) -> str:
    """
    Turns messy source column names into safe, consistent SQL column names:
    lowercase, spaces/hyphens -> underscores, strips anything that isn't
    alphanumeric or underscore.
    """
    col = col.strip().lower()
    col = re.sub(r"[\s\-]+", "_", col)
    col = re.sub(r"[^a-z0-9_]", "", col)
    return col or "unnamed_col"


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Applies general-purpose cleaning that's safe for any of the 3 datasets."""
    df = df.copy()

    # 1. Normalize column names.
    df.columns = [clean_column_name(c) for c in df.columns]

    # 2. Strip whitespace from all text cells and convert empty strings to NaN.
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": None, "nan": None, "None": None})

    # 3. Try to convert text columns that are actually numeric (e.g. "1,200")
    #    into real numeric columns, but only if EVERY non-null value converts
    #    cleanly — this avoids corrupting genuinely mixed text columns.
    for col in df.select_dtypes(include=["object", "string"]).columns:
        stripped = df[col].dropna().astype(str).str.replace(",", "", regex=False)
        if len(stripped) == 0:
            continue
        is_int = stripped.str.match(r"^-?\d+$").all()
        is_float = stripped.str.match(r"^-?\d+\.\d+$").all()
        if is_int:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", "", regex=False),
                errors="coerce",
            ).astype("Int64")
        elif is_float:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", "", regex=False),
                errors="coerce",
            )

    # 4. Drop exact duplicate rows.
    df = df.drop_duplicates()

    return df


def build_one_database(short_name: str, csv_file: str, db_file: str, table_name: str):
    csv_path = os.path.join(DATA_DIR, csv_file)
    db_path = os.path.join(DB_DIR, db_file)

    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"Missing {csv_path}. Run scripts/download_and_inspect.py first."
        )

    df = pd.read_csv(csv_path)
    df = clean_dataframe(df)

    # Overwrite any existing db file so this script is safely re-runnable.
    if os.path.exists(db_path):
        os.remove(db_path)

    engine = create_engine(f"sqlite:///{db_path}")
    df.to_sql(table_name, engine, if_exists="replace", index=False)

    print(f"\nBuilt {db_path}  (table: {table_name}, rows: {len(df)})")
    print("Columns:", list(df.columns))

    # Print the real CREATE TABLE statement SQLite generated, so we can see
    # the actual inferred column types (TEXT / INTEGER / REAL).
    conn = sqlite3.connect(db_path)
    schema_sql = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
        (table_name,),
    ).fetchone()[0]
    print("Schema:\n", schema_sql)
    conn.close()


def main():
    for short_name, (csv_file, db_file, table_name) in TABLES.items():
        build_one_database(short_name, csv_file, db_file, table_name)

    print("\nAll three SQLite databases built successfully in ./db/")


if __name__ == "__main__":
    main()


### `agent/main_agent.py` — the routing agent

In [ ]:
%%writefile agent/main_agent.py
"""
STEP 6: The main routing agent.

A NOTE ON WHICH AGENT API WE USE
----------------------------------
LangChain's older `create_tool_calling_agent` + `AgentExecutor` combo (from
`langchain.agents`) still works, but the LangChain team now recommends
LangGraph's `create_react_agent` as the standard way to build tool-calling
agents — it's actively maintained, has built-in streaming/memory support,
and is a straight drop-in for this kind of "route to the right tool" use
case. That's what we use here. (If you specifically need the legacy
AgentExecutor for a course requirement, the swap is small — see the comment
at the bottom of this file.)

ROUTING LOGIC
--------------
We don't hand-write if/else routing rules. Instead, the LLM itself decides
which tool to call based on:
  1. Each tool's docstring/description (see tools/db_tools.py and
     tools/web_search_tool.py) — this is the PRIMARY signal.
  2. The system prompt below, which gives a few worked examples to reinforce
     the boundary between "structured data lookup" and "general knowledge".
"""

import os
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

from tools.db_tools import institutions_db_tool, hospitals_db_tool, restaurants_db_tool
from tools.web_search_tool import web_search_tool

SYSTEM_PROMPT = """You are a helpful assistant that answers questions about
Bangladesh using four tools:

1. institutions_db_tool — structured data about universities, colleges, and
   government institutions (names, locations, types).
2. hospitals_db_tool — structured data about hospitals (names, locations,
   beds, doctors, facilities).
3. restaurants_db_tool — structured data about restaurants (names,
   locations, cuisine, ratings).
4. web_search_tool — general knowledge: definitions, policy, history,
   culture — anything NOT found in the three databases above.

ROUTING RULES:
- If the question asks to COUNT, LIST, or FILTER specific hospitals,
  institutions, or restaurants (e.g. mentions a city, a rating threshold, a
  bed count, an institution type) -> use the matching DB tool.
- If the question asks "what is", "what is the role of", "explain", or asks
  about policy/history/culture/definitions -> use web_search_tool.
- Only call ONE tool per question unless the question genuinely has two
  distinct parts that need two different tools.
- Base your final answer ONLY on what the tool returned. If a tool returns
  no data, say so honestly instead of guessing.

WORKED EXAMPLES:
- "How many hospitals are in Dhaka?" -> hospitals_db_tool
- "List restaurants in Chittagong with a rating above 4." -> restaurants_db_tool
- "What government universities are in Sylhet?" -> institutions_db_tool
- "What is the role of DGHS in Bangladesh?" -> web_search_tool
- "What is Bangladesh's national health policy?" -> web_search_tool
"""


def build_agent():
    """
    Creates and returns a ready-to-invoke LangGraph agent with all four
    tools registered and the routing system prompt attached.
    """
    llm = ChatOpenAI(model=os.getenv("AGENT_MODEL", "gpt-4o"), temperature=0)

    tools = [
        institutions_db_tool,
        hospitals_db_tool,
        restaurants_db_tool,
        web_search_tool,
    ]

    agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
    return agent


def ask(agent, question: str) -> dict:
    """
    Sends one question to the agent and returns the full result dict
    (including the tool-call trace, useful for STEP 7 testing/debugging).
    """
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result


def final_answer_text(result: dict) -> str:
    """Extracts just the final assistant message text from an agent result."""
    return result["messages"][-1].content


# ---------------------------------------------------------------------------
# LEGACY SWAP NOTE: if you must use the older AgentExecutor pattern instead
# of LangGraph, replace build_agent() with:
#
#   from langchain.agents import create_tool_calling_agent, AgentExecutor
#   from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#
#   prompt = ChatPromptTemplate.from_messages([
#       ("system", SYSTEM_PROMPT),
#       ("human", "{input}"),
#       MessagesPlaceholder("agent_scratchpad"),
#   ])
#   agent = create_tool_calling_agent(llm, tools, prompt)
#   return AgentExecutor(agent=agent, tools=tools, verbose=True)
#
# and call it with agent_executor.invoke({"input": question}) instead of
# the ask()/final_answer_text() helpers above.
# ---------------------------------------------------------------------------


## Step 4 — Download and inspect the datasets

This runs the exact same script as the repo's `scripts/download_and_inspect.py`.
Read the printed schema/sample rows/messy-data warnings before moving on —
they tell you the *real* column names, which the rest of the pipeline relies on.


In [ ]:
!python scripts/download_and_inspect.py


## Step 5 — Build the SQLite databases

In [ ]:
!python scripts/build_databases.py


## Step 6 — Build the agent and ask it questions

This builds the LangGraph routing agent from `agent/main_agent.py` (the same
code used by `main.py` in the repo) and runs it right here in the notebook.


In [ ]:
import sys
sys.path.append(os.getcwd())  # so "import agent...", "import tools..." resolve

from agent.main_agent import build_agent, ask, final_answer_text

agent = build_agent()
print("Agent ready.")


## Step 7 — Test the 5 example queries

For each question we print:
- the tool(s) the agent actually called (the routing decision)
- the final natural-language answer


In [ ]:
TEST_CASES = [
    ("How many hospitals are in Dhaka?", "hospitals_db_tool"),
    ("List restaurants in Chittagong with a rating above 4.", "restaurants_db_tool"),
    ("What government universities are in Sylhet?", "institutions_db_tool"),
    ("What is the role of DGHS in Bangladesh?", "web_search_tool"),
    ("What is Bangladesh's national health policy?", "web_search_tool"),
]

def get_called_tools(result):
    called = []
    for msg in result["messages"]:
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            called.extend(call["name"] for call in tool_calls)
    return called

for question, expected_tool in TEST_CASES:
    print("=" * 70)
    print("QUESTION:", question)
    print("EXPECTED TOOL:", expected_tool)

    result = ask(agent, question)
    called = get_called_tools(result)
    print("ACTUAL TOOL(S) CALLED:", called)
    print("ROUTING CORRECT:", "YES" if expected_tool in called else "NO")
    print("ANSWER:", final_answer_text(result))
    print()


## Step 8 — Ask your own question

Change the `my_question` string below and re-run this cell to chat with
the agent interactively, right in the notebook.


In [ ]:
my_question = "Which hospitals in Chittagong have more than 100 beds?"

result = ask(agent, my_question)
print("Tool(s) called:", get_called_tools(result))
print("Answer:", final_answer_text(result))


## Notes

- This notebook is functionally identical to running `main.py` /
  `scripts/test_queries.py` from the full repo — it just inlines every file
  with `%%writefile` so it works standalone in Colab with zero setup beyond
  API keys.
- All generated SQL is read-only and validated by `tools/sql_safety.py`
  before it ever touches a database (see that cell above for details).
- For a persistent chat UI instead of notebook cells, see `streamlit_app.py`
  and `docker-compose.yml` in the full repo — run `docker compose up --build`
  and open `http://localhost:8501`.
